In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import calendar
import math

In [3]:
def plot_monthly_yearly_horizontal_bars(
    df,
    year_col='Year',
    month_col='Month',
    cases_col='Case_TOTAL',
    bar_height_scale=1.0,
    spacing_factor=1.5,
    title='Monthly Malaria Cases per Year (Horizontal)',
    xlabel='Cases',
    ylabel='Year',
    text_font_size = 7,
    x_ticks_rounding_factor = -2,
    x_ticks_percentage_split = 5,
    figsize=(15, 10),
    show_values=True
):
    """
    Plots a horizontal grouped bar chart of monthly cases per year.

    Parameters:
        df (pd.DataFrame): Input DataFrame.
        year_col (str): Column name for the year.
        month_col (str): Column name for the month (must be numeric 1–12).
        cases_col (str): Column name for the total cases.
        bar_height_scale (float): Scale of bar height (default 1.0 = standard).
        spacing_factor (float): Vertical spacing between year groups.
        title (str): Plot title.
        xlabel (str): X-axis label.
        ylabel (str): Y-axis label.
        figsize (tuple): Figure size.
        show_values (bool): Whether to label bar ends with case numbers.
    """

    # ────────────────
    # 1. Aggregate monthly totals
    # ────────────────
    df_monthly = (df.groupby([year_col, month_col])[cases_col]
                    .sum()
                    .reset_index())

    # Convert numeric month to abbreviated name
    df_monthly['Month_name'] = df_monthly[month_col].apply(lambda m: calendar.month_abbr[m])

    # Pivot: rows = year, columns = months
    month_order = list(calendar.month_abbr[1:])  # Jan to Dec
    pivot_df = (df_monthly.pivot(index=year_col, columns='Month_name', values=cases_col)
                          .reindex(columns=month_order)
                          .sort_index())

    # ────────────────
    # 2. Plotting
    # ────────────────
    fig, ax = plt.subplots(figsize=figsize)

    years = pivot_df.index.tolist()
    months = pivot_df.columns.tolist()
    num_months = len(months)

    bar_height = bar_height_scale / num_months
    y_positions = np.arange(len(years)) * spacing_factor

    for i, month in enumerate(months):
        month_cases = pivot_df[month]
        offsets = y_positions + i * bar_height
        bars = ax.barh(offsets, month_cases, height=bar_height, label=month)

        if show_values:
            for bar in bars:
                width = bar.get_width()
                y = bar.get_y() + bar.get_height() / 2
                ax.text(width + max(pivot_df.max()) * 0.01, y, f'{int(width):,}', va='center', fontsize=text_font_size)

    
    ax.set_xticks(np.arange(0, max(pivot_df.max()) + math.ceil(10*max(pivot_df.max())/100), step=round((x_ticks_percentage_split*max(pivot_df.max())/100), x_ticks_rounding_factor)))
    plt.xticks(rotation=45, ha='right')

    ax.set_yticks(y_positions + bar_height * (num_months - 1) / 2)
    ax.set_yticklabels(years, fontsize=14)

    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel(ylabel, fontsize=16)
    ax.set_title(title, fontsize=20)

    ax.legend(title='Month', bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.grid(axis='x', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()